## AGENTIC RAG WITH LangGraph

In [1]:
import sys
!{sys.executable} -m pip install tavily-python -q

import os # for environment variable management
from typing import TypedDict # for defining a structured type for the agent's state
from langgraph.graph import StateGraph, END # for creating a state graph to manage the agent's states and transitions
from langchain_anthropic import ChatAnthropic # for using Anthropic's language model
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader, CSVLoader, UnstructuredMarkdownLoader, UnstructuredPowerPointLoader # for loading different document formats
from langchain_community.tools.tavily_search import TavilySearchResults # for web search fallback

from langchain_community.embeddings import SentenceTransformerEmbeddings  # for generating embeddings from text. Takes care of retrieving from hugging face directly, so you don't have to worry about it.
#SentenceTransformerEmbeddings()
#        ↓
#Downloads "sentence-transformers/all-mpnet-base-v2" from HuggingFace
#        ↓
#Runs locally on your machine

from langchain_community.vectorstores import FAISS # for vector storage and retrieval
from langchain_text_splitters import RecursiveCharacterTextSplitter # for splitting documents into chunks
from langchain_core.documents import Document # for type hinting
import glob # for file searching

c:\Users\aurin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\aurin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [3]:
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"]   = os.getenv("TAVILY_API_KEY")

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0.7) # temperature 0 for deterministic output. 0 is for more deterministic output, while higher values (e.g., 0.7) can introduce more creativity and variability in responses.

embeddings = SentenceTransformerEmbeddings()

C:\Users\Default\AppData\Local\Temp\ipykernel_28256\1348797104.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings()
C:\Users\Default\AppData\Local\Temp\ipykernel_28256\1348797104.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = SentenceTransformerEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 901.86it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers

In [4]:
llm

ChatAnthropic(profile={'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'structured_output': False}, model='claude-sonnet-4-6', max_tokens=64000, temperature=0.7, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

## STATE DEFINITION

In [5]:
class AgentState(TypedDict):
    query: str
    retrieved_docs: list[Document]
    answer: str
    route_decision: str         # "local", "web", or "general" — set by decide
    retrieval_count: int        # tracks retrieval attempts to cap the loop
    chat_history: list[str]     # stores prior turns for multi-turn conversations

In [6]:
import sys
!{sys.executable} -m pip install docx2txt -q

LOADERS = {
    ".pdf":  PyPDFLoader,
    ".txt":  TextLoader,
    ".docx": Docx2txtLoader,
    ".doc":  Docx2txtLoader,
    ".csv":  CSVLoader,
    ".md":   UnstructuredMarkdownLoader,
    ".pptx": UnstructuredPowerPointLoader
}

def load_document(file_path: str) -> list[Document]:
    ext = os.path.splitext(file_path)[1].lower()
    loader_cls = LOADERS.get(ext)
    if not loader_cls:
        print(f"  [SKIP] Unsupported type: {file_path}")
        return []
    try:
        docs = loader_cls(file_path).load()
        for doc in docs:
            doc.metadata["source"] = os.path.basename(file_path)
        print(f"  [OK] {os.path.basename(file_path)} → {len(docs)} page(s)")
        return docs
    except Exception as e:
        print(f"  [ERROR] {os.path.basename(file_path)}: {e}")
        return []

def build_vectorstore(folder_path: str, chunk_size: int = 500, chunk_overlap: int = 100):
    file_paths = []
    for ext in LOADERS:
        file_paths.extend(glob.glob(os.path.join(folder_path, f"**/*{ext}"), recursive=True))

    if not file_paths:
        raise FileNotFoundError(f"No supported documents found in '{folder_path}'")

    print(f"Found {len(file_paths)} file(s). Loading...")
    all_docs = []
    for path in file_paths:
        all_docs.extend(load_document(path))

    if not all_docs:
        raise ValueError("No content could be extracted from the documents.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(all_docs)
    print(f"\nSplit into {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")

    vs = FAISS.from_documents(chunks, embeddings)
    print("Vectorstore ready.")
    return vs


FAISS_INDEX_PATH = "faiss_index"

# Load from disk if index already exists, otherwise build and save
if os.path.exists(FAISS_INDEX_PATH):
    print("Loading vectorstore from disk...")
    vectorstore = FAISS.load_local(FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
    print("Vectorstore loaded.")
else:
    vectorstore = build_vectorstore("documents")
    vectorstore.save_local(FAISS_INDEX_PATH)
    print(f"Vectorstore saved to '{FAISS_INDEX_PATH}'.")

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20}
)

Loading vectorstore from disk...
Vectorstore loaded.


## AGENT FUNCTION

In [7]:
MAX_RETRIEVAL_ATTEMPTS = 2

# Build document source list from vectorstore so decide() knows what's available
_indexed_sources = {
    doc.metadata.get("source", "unknown")
    for doc in vectorstore.docstore._dict.values()
}

_folder_sources = {
    os.path.basename(f)
    for ext in LOADERS
    for f in glob.glob(os.path.join("documents", f"**/*{ext}"), recursive=True)
}

_available_sources = list(_indexed_sources)

_unindexed = _folder_sources - _indexed_sources
if _unindexed:
    print(f"WARNING: These files are in documents/ but NOT in the index: {_unindexed}")
    print("Delete faiss_index/ and restart to include them.")

# ── NODE FUNCTIONS ─────────────────────────────────────────────────────────────

def decide(state: AgentState) -> AgentState:
    sources_list = "\n".join(f"- {s}" for s in _available_sources)
    prompt = f"""You are a routing assistant for a document Q&A system.
The following documents are available to search:
{sources_list}

Classify the user's question into ONE of these categories:
- 'local'  : the question relates to the documents listed above
- 'web'    : the question asks for factual information not covered by the documents (recent events, current statistics, news, specific technical info the LLM may not know)
- 'general': the question can be answered from general knowledge without any search (math, basic definitions, casual conversation)

Answer ONLY 'local', 'web', or 'general'.

Question: {state['query']}"""
    response = llm.invoke(prompt)
    decision = response.content.strip().lower()
    if decision not in ("local", "web", "general"):
        decision = "general"
    state["route_decision"] = decision
    state["retrieval_count"] = 0
    print(f"[decide] route={state['route_decision']}")
    return state


def retrieve(state: AgentState) -> AgentState:
    docs = retriever.invoke(state["query"])
    state["retrieved_docs"] = docs
    state["retrieval_count"] = state.get("retrieval_count", 0) + 1
    print(f"[retrieve] got {len(docs)} chunks (attempt {state['retrieval_count']})")
    return state


def query(state: AgentState) -> AgentState:
    context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
    prompt = f"""The retrieved documents did not fully answer the question.
Rewrite the question to be more specific so better documents can be found.
Return ONLY the rewritten question.

Original question: {state['query']}

Retrieved context:
{context}

Rewritten question:"""
    response = llm.invoke(prompt)
    state["query"] = response.content.strip()
    print(f"[query] reformulated → '{state['query']}'")
    return state


def web_search(state: AgentState) -> AgentState:
    print("[web_search] searching web...")
    tool = TavilySearchResults(max_results=4)
    results = tool.invoke(state["query"])
    state["retrieved_docs"] = [
        Document(page_content=r["content"], metadata={"source": r["url"]})
        for r in results
    ]
    print(f"[web_search] got {len(state['retrieved_docs'])} results")
    return state


def generate(state: AgentState) -> AgentState:
    history_block = ""
    if state.get("chat_history"):
        history_block = "Conversation so far:\n" + "\n".join(state["chat_history"]) + "\n\n"

    if state.get("retrieved_docs"):
        context = "\n\n".join(
            f"[{doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
            for doc in state["retrieved_docs"]
        )
        prompt = f"""{history_block}Answer the question using the provided context only.
Cite the source in brackets for each fact you use (e.g. [LOGISTIC_VS_LINEAR.pdf] or [https://...]).
If the context does not contain enough information, say so clearly.

Context:
{context}

Question: {state['query']}
Answer:"""
    else:
        prompt = f"""{history_block}Answer the following question using your general knowledge.

Question: {state['query']}
Answer:"""

    response = llm.invoke(prompt)
    state["answer"] = response.content.strip()
    state["chat_history"] = state.get("chat_history", []) + [
        f"User: {state['query']}",
        f"Assistant: {state['answer']}",
    ]
    print(f"[generate] done.")
    return state


# ── ROUTING ────────────────────────────────────────────────────────────────────

def route_decide(state: AgentState) -> str:
    mapping = {"local": "retrieve", "web": "web_search", "general": "generate"}
    return mapping[state["route_decision"]]


def route_after_retrieve(state: AgentState) -> str:
    if state["retrieval_count"] >= MAX_RETRIEVAL_ATTEMPTS:
        print(f"[route] max attempts reached → web_search")
        return "web_search"

    if not state.get("retrieved_docs"):
        return "query"

    top_doc = state["retrieved_docs"][0].page_content
    prompt = f"""Is this document relevant to answering the question?
Answer ONLY 'yes' or 'no'.

Question: {state['query']}
Document: {top_doc}"""
    response = llm.invoke(prompt)
    decision = "generate" if response.content.strip().lower().startswith("yes") else "query"
    print(f"[route_after_retrieve] → {decision}")
    return decision


# ── BUILD GRAPH ──────────────────────────────────────────────────────────────

def build_agent():
    graph = StateGraph(AgentState)

    graph.add_node("decide",     decide)
    graph.add_node("retrieve",   retrieve)
    graph.add_node("query",      query)
    graph.add_node("web_search", web_search)
    graph.add_node("generate",   generate)

    graph.set_entry_point("decide")

    graph.add_conditional_edges("decide", route_decide, {
        "retrieve":   "retrieve",    # route_decision="local"
        "web_search": "web_search",  # route_decision="web"
        "generate":   "generate",    # route_decision="general"
    })
    graph.add_conditional_edges("retrieve", route_after_retrieve, {
        "generate":   "generate",
        "query":      "query",
        "web_search": "web_search",
    })
    graph.add_edge("query",      "retrieve")
    graph.add_edge("web_search", "generate")
    graph.add_edge("generate",   END)

    return graph.compile()


agent = build_agent()
print("Agent compiled.")


# ── RUNNER ───────────────────────────────────────────────────────────────────

_chat_history: list[str] = []

def run_agent(question: str, reset_history: bool = False) -> str:
    global _chat_history
    if reset_history:
        _chat_history = []
    result = agent.invoke({
        "query":           question,
        "retrieved_docs":  [],
        "answer":          "",
        "route_decision":  "general",
        "retrieval_count": 0,
        "chat_history":    _chat_history.copy(),
    })
    _chat_history = result["chat_history"]
    return result["answer"]

Agent compiled.


## TESTING

In [8]:
answer = run_agent("What is the difference between logistic and linear regression?")
print(answer)


[decide] route=local
[retrieve] got 4 chunks (attempt 1)
[route_after_retrieve] → query
[query] reformulated → 'How does logistic regression differ from linear regression in terms of their cost functions, decision boundaries, and suitability for classification tasks?'
[retrieve] got 4 chunks (attempt 2)
[route] max attempts reached → web_search
[web_search] searching web...


C:\Users\Default\AppData\Local\Temp\ipykernel_28256\2430859360.py:76: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=4)


[web_search] got 4 results
[generate] done.
## Logistic Regression vs Linear Regression: Key Differences

### Cost Functions

Linear regression typically uses **Ordinary Least Squares (OLS)** to minimize the mean squared error, while logistic regression uses **Maximum Likelihood Estimation** to optimize its parameters [https://towardsdatascience.com/linear-regression-vs-logistic-regression-ols-maximum-likelihood-estimation-gradient-descent-bcfac2c7b8e4/]. In both cases, when an analytical solution is not available, **Gradient Descent** can be used as a numerical approximation approach to find optimal parameters, though it has limitations such as not working with non-convex functions [https://towardsdatascience.com/linear-regression-vs-logistic-regression-ols-maximum-likelihood-estimation-gradient-descent-bcfac2c7b8e4/].

### Decision Boundaries

- In **linear regression**, the model fits a line (or hyperplane) to best represent continuous output values [https://community.deeplearning.a

In [9]:
_chat_history.clear()

print(run_agent("What was the average age of Titanic passengers?"))
#print("---")
#print(run_agent("Can you give me an example of that?"))
#print("---")
#print(run_agent("How does it compare to linear regression?"))

[decide] route=web
[web_search] searching web...
[web_search] got 4 results
[generate] done.
The provided context does not contain a specific figure for the overall average age of all Titanic passengers. 

However, the sources do provide some related age information:

- The average age of **adult passengers** (ages 18-65) on the ship was **a little over 30 years** [https://rkursus.github.io/2020/projektid/group1/Titanic-Passengers-List.pdf].
- The average age of **female passengers** was slightly lower than that of male passengers [https://rkursus.github.io/2020/projektid/group1/Titanic-Passengers-List.pdf].
- The group aged **65 and older consisted exclusively of men** [https://rkursus.github.io/2020/projektid/group1/Titanic-Passengers-List.pdf].

A precise overall average age for all passengers is not stated in the provided context.
